# OrbitalNet — Exploration Notebook

Optional notebook for interactively exploring the dataset and trained models, and for generating figures for the IEEE paper / viva presentation.

**Prerequisites:** run this from the project root (so `src/` is importable), and make sure you've already run:
```
python scripts/generate_data.py
python scripts/train_pinn.py
python scripts/train_baseline.py
```

In [ ]:
import os, sys
sys.path.append(os.path.abspath('..'))

import numpy as np
import torch
import matplotlib.pyplot as plt

from src.data_generation import load_dataset, train_val_test_split
from src.model import OrbitalMLP
from src.physics import simulate_trajectory
from src.evaluate import predict, evaluate_basic, evaluate_noise_robustness, evaluate_long_horizon
from src.visualize import plot_trajectory_matplotlib, plot_loss_curve

%matplotlib inline

## 1. Load the dataset and inspect a few orbits

In [ ]:
dataset = load_dataset('../data/orbitalnet_dataset.npz')
splits = train_val_test_split(dataset)
print('Total samples:', dataset['X'].shape[0])
print({k: v['X'].shape[0] for k, v in splits.items()})

In [ ]:
# Plot a handful of raw simulated orbits to sanity-check diversity (circular vs elliptical)
fig, ax = plt.subplots(figsize=(6, 6))
for tid in range(8):
    mask = dataset['traj_id'] == tid
    xy = dataset['Y'][mask][:, 0:2]
    ax.plot(xy[:, 0], xy[:, 1])
ax.scatter([0], [0], c='orange', marker='*', s=200)
ax.set_aspect('equal')
ax.set_title('Sample of simulated ground-truth orbits')
plt.show()

## 2. Load trained models

In [ ]:
def load_trained(model_path, stats_path):
    model = OrbitalMLP()
    model.load_state_dict(torch.load(model_path, map_location='cpu', weights_only=True))
    model.eval()
    stats_npz = np.load(stats_path)
    stats = {k: stats_npz[k] for k in stats_npz.files}
    return model, stats

pinn_model, pinn_stats = load_trained('../models/pinn_model.pt', '../models/pinn_stats.npz')
baseline_model, baseline_stats = load_trained('../models/baseline_model.pt', '../models/baseline_stats.npz')
print('Models loaded.')

## 3. Loss convergence curves

In [ ]:
import json
with open('../models/pinn_history.json') as f:
    pinn_history = json.load(f)
plot_loss_curve(pinn_history, title='PINN training loss convergence')
plt.show()

## 4. Compare PINN vs baseline on a test orbit

In [ ]:
test_traj_ids = np.unique(splits['test']['traj_id'])
tid = test_traj_ids[0]
state0 = dataset['init_states'][tid]
period = dataset['periods'][tid]
t_eval = np.linspace(0, 1.15 * period, 200)
true_traj = simulate_trajectory(state0, t_eval)

X_query = np.zeros((200, 5))
X_query[:, 0:4] = state0
X_query[:, 4] = t_eval

pinn_pred = predict(pinn_model, X_query, pinn_stats)
baseline_pred = predict(baseline_model, X_query, baseline_stats)

plot_trajectory_matplotlib(true_traj, pinn_pred, title='PINN: predicted vs true orbit')
plt.show()
plot_trajectory_matplotlib(true_traj, baseline_pred, title='Baseline MLP: predicted vs true orbit')
plt.show()

## 5. Full comparative evaluation table

In [ ]:
for name, model, stats in [('PINN', pinn_model, pinn_stats), ('Baseline', baseline_model, baseline_stats)]:
    basic = evaluate_basic(model, splits, stats)
    noise = evaluate_noise_robustness(model, splits, stats)
    long_horizon = evaluate_long_horizon(model, splits, stats, dataset)
    print(f'--- {name} ---')
    print('Test MSE/MAE:', basic)
    print('Noise robustness:', noise)
    print('Long-horizon:', long_horizon)
    print()